# Agent: Humans Producer (Phase 1, No MQTT)

This notebook implements only Phase 1:
- local movement in a 1000x1000 meter area
- boundary reflection with overshoot mirroring
- illegal event generation (`event_type=\"illegal_act\"`)
- max 1 event per human per step
- 1 Hz simulation cadence

No MQTT publish/subscribe is included in this phase.


In [ ]:
# Setup / imports
from __future__ import annotations

import colorsys
import math
import random
import time
from dataclasses import dataclass
from typing import Any

from IPython.display import display

from anymap_ts import Map
from simulated_city.config import load_config


In [ ]:
# Config load + Phase 1 constants
cfg = load_config()

# Parken Stadion (Copenhagen) center
PARKEN_LAT = 55.7029
PARKEN_LON = 12.5720

AREA_MIN = 0.0
AREA_MAX = 1000.0
STEP_LENGTH_M = 7.0
STEP_DELAY_S = 1.0
HUMAN_COUNT = 10
EVENT_PROBABILITY = 0.25
TOTAL_STEPS = 60
EVENT_TYPE = "illegal_act"

print(f"Config loaded. Primary MQTT host: {cfg.mqtt.host}")
print(f"Phase 1 run: humans={HUMAN_COUNT}, steps={TOTAL_STEPS}, delay={STEP_DELAY_S}s")


In [ ]:
# Helper functions: local meter coordinates, boundary reflection, and colors
def local_xy_to_lnglat(x_m: float, y_m: float, center_lat: float, center_lon: float) -> tuple[float, float]:
    """Approximate local meters (x east, y north) to (lng, lat)."""
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lon = 111_320.0 * math.cos(math.radians(center_lat))
    lat = center_lat + (y_m / meters_per_deg_lat)
    lon = center_lon + (x_m / meters_per_deg_lon)
    return lon, lat


def reflect_axis(position: float, delta: float, low: float = AREA_MIN, high: float = AREA_MAX) -> tuple[float, float]:
    """Reflect one axis and mirror overshoot into [low, high]."""
    new_position = position + delta
    new_delta = delta

    while new_position < low or new_position > high:
        if new_position > high:
            overshoot = new_position - high
            new_position = high - overshoot
            new_delta = -new_delta
        elif new_position < low:
            overshoot = low - new_position
            new_position = low + overshoot
            new_delta = -new_delta

    return new_position, new_delta


def generate_unique_colors(count: int) -> list[str]:
    """Create random-but-unique colors for human markers."""
    hues = [index / count for index in range(count)]
    random.shuffle(hues)

    colors: list[str] = []
    for hue in hues:
        r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
        colors.append(f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}")

    return colors


In [ ]:
# Human model + initialization
@dataclass
class Human:
    human_id: str
    name: str
    x: float
    y: float
    marker_color: str
    criminal_count: int = 0


NAMES = [
    "Ava Jensen",
    "Liam Nielsen",
    "Emma Larsen",
    "Noah Pedersen",
    "Sofia Madsen",
    "Oliver Christensen",
    "Clara Andersen",
    "William Thomsen",
    "Freja Rasmussen",
    "Lucas Mortensen",
]

colors = generate_unique_colors(HUMAN_COUNT)
humans: list[Human] = []
for index in range(HUMAN_COUNT):
    humans.append(
        Human(
            human_id=NAMES[index],
            name=NAMES[index],
            x=random.uniform(AREA_MIN, AREA_MAX),
            y=random.uniform(AREA_MIN, AREA_MAX),
            marker_color=colors[index],
        )
    )

print("Initialized humans:")
for human in humans:
    print(f"- {human.human_id}: x={human.x:.1f}, y={human.y:.1f}, color={human.marker_color}")


In [ ]:
# Map setup (anymap)
m = Map(center=(PARKEN_LON, PARKEN_LAT), zoom=15.8, height="650px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")

# Render initial markers
for human in humans:
    lng, lat = local_xy_to_lnglat(human.x - 500.0, human.y - 500.0, PARKEN_LAT, PARKEN_LON)
    m.add_marker(lng, lat, name=human.human_id, color=human.marker_color, popup=human.name)

display(m)
print("Map displayed. Run the simulation cell to start.")


In [ ]:
# 1 Hz simulation loop: movement + event generation (no MQTT)
events_by_step: dict[int, list[dict[str, Any]]] = {}

for step in range(TOTAL_STEPS):
    step_started = time.perf_counter()
    step_events: list[dict[str, Any]] = []

    for human in humans:
        # Continuous movement with random direction
        angle = random.uniform(0.0, 2.0 * math.pi)
        dx = STEP_LENGTH_M * math.cos(angle)
        dy = STEP_LENGTH_M * math.sin(angle)

        # Reflect independently per axis with mirrored overshoot
        human.x, _ = reflect_axis(human.x, dx)
        human.y, _ = reflect_axis(human.y, dy)

        # Update marker position
        lng, lat = local_xy_to_lnglat(human.x - 500.0, human.y - 500.0, PARKEN_LAT, PARKEN_LON)
        m.remove_marker(human.human_id)
        m.add_marker(lng, lat, name=human.human_id, color=human.marker_color, popup=human.name)

        # Illegal event generation (max 1 per human per step)
        if random.random() < EVENT_PROBABILITY:
            step_events.append(
                {
                    "step": step,
                    "event_id": f"{step}:{human.human_id}",
                    "human_id": human.human_id,
                    "name": human.name,
                    "x": human.x,
                    "y": human.y,
                    "event_type": EVENT_TYPE,
                }
            )

    events_by_step[step] = step_events
    print(f"step={step:03d} events={len(step_events)}")

    elapsed = time.perf_counter() - step_started
    time.sleep(max(0.0, STEP_DELAY_S - elapsed))

print("Phase 1 simulation complete.")


In [ ]:
# Quick verification checks for Phase 1
all_event_ids = [event["event_id"] for events in events_by_step.values() for event in events]
unique_event_ids = set(all_event_ids)

print(f"Total events: {len(all_event_ids)}")
print(f"Unique event_ids: {len(unique_event_ids)}")
print(f"event_id uniqueness ok: {len(all_event_ids) == len(unique_event_ids)}")

if all_event_ids:
    print("Sample event_ids:")
    for event_id in list(unique_event_ids)[:10]:
        print(" -", event_id)
